# Question preview

Prints every question to Json

In [1]:
%pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
QUESTIONS_FILE = "eval_questions.xlsx"   # path to your xlsx, tsv or csv
QUESTIONS_JSON = "questions.json"        # parsed output that prototype_nav reads

In [3]:
import re, json
from pathlib import Path
import pandas as pd

LETTERS="ABCDE"

def _blank(v):
    if v is None: return True
    s=str(v).strip(); return s=="" or s.lower()=="nan"

# pull a..e out of a cell that may hold one choice or all of them, one per line
def _parse_options_cell(text):
    opts={}
    for line in re.split(r"[\r\n;]+", str(text)):
        mm=re.match(r"^\s*\(?\s*([A-Ea-e])\s*[).:\-\u2013]\s*(.+)$", line.strip())
        if mm and mm.group(1).upper() not in opts: opts[mm.group(1).upper()]=mm.group(2).strip()
    return opts

# the correct answer may be one letter or several like "a, c, d"; grab every standalone a..e letter
def _parse_answers(cell):
    found={m.upper() for m in re.findall(r"(?<![A-Za-z])([A-Ea-e])(?![A-Za-z])", str(cell))}
    return [L for L in LETTERS if L in found]

def _match(df):
    norm={re.sub(r"[^a-z]","",str(c).lower()):c for c in df.columns}
    def col(*names):
        for n in names:
            if n in norm: return norm[n]
        return None
    return dict(q=col("question","q","prompt","stem"),
                ans=col("correctanswer","answer","correct","gold","label","key"),
                idc=col("number","id","qid","index"),
                diff=col("difficulty","level"),
                combined=col("mcoptions","options","choices","mcq","multiplechoiceoptions"),
                A=col("a","optiona","choicea"),B=col("b","optionb","choiceb"),
                C=col("c","optionc","choicec"),D=col("d","optiond","choiced"),E=col("e","optione","choicee"))

def _find_header(raw):
    best,score=0,-1
    for r in range(min(8,len(raw))):
        cells=[re.sub(r"[^a-z]","",str(x).lower()) for x in raw.iloc[r].tolist()]
        s=sum(any(k in c for k in ("question","answer","option","number","choice")) for c in cells)
        if s>score: score,best=s,r
    return best

def _prep(raw):
    hr=_find_header(raw)
    df=raw.iloc[hr+1:].copy(); df.columns=[str(c) for c in raw.iloc[hr].tolist()]
    return df.reset_index(drop=True)

# one sheet -> questions; handles split a..e columns, one cell of choices, or choices merged across rows
def _rows(df, sheet, out, skipped, seen):
    m=_match(df)
    has_split=all(m[k] for k in ("A","B","C","D"))
    if not (m["q"] and m["ans"] and (has_split or m["combined"])):
        skipped.append({"sheet":sheet,"reason":"no usable columns","preview":str(list(df.columns))[:80]}); return
    def add(idv,stem,opts,ansv,diffv):
        opts={L:opts[L] for L in LETTERS if L in opts and not _blank(opts.get(L))}
        if _blank(stem): return
        if len(opts)<2: skipped.append({"sheet":sheet,"reason":"fewer than 2 options","preview":str(stem)[:70]}); return
        answers=[a for a in _parse_answers(ansv)]
        if not answers: skipped.append({"sheet":sheet,"reason":"no answer letter","preview":str(stem)[:70]}); return
        qid=str(idv).strip() if not _blank(idv) else f"{sheet}_{len(out)+1}"
        while qid in seen: qid+="_x"
        seen.add(qid)
        out.append({"qid":qid,"sheet":sheet,"stem":str(stem).strip(),"options":opts,
                    "answer":answers[0],"answers":answers,
                    "difficulty":"" if _blank(diffv) else str(diffv).strip()})
    if has_split:
        for _,r in df.iterrows():
            opts={L:str(r[m[L]]) for L in LETTERS if m.get(L)}
            add(r[m["idc"]] if m["idc"] else None, r[m["q"]], opts, r[m["ans"]],
                r[m["diff"]] if m["diff"] else None)
    else:
        keycol=m["idc"] or m["q"]
        grp=df[keycol].apply(lambda x: not _blank(x)).cumsum()
        for _,sub in df.groupby(grp):
            def first(c):
                for v in sub[c].tolist():
                    if not _blank(v): return v
                return None
            lines="\n".join(str(v) for v in sub[m["combined"]].tolist() if not _blank(v))
            add(first(m["idc"]) if m["idc"] else None, first(m["q"]), _parse_options_cell(lines), first(m["ans"]),
                first(m["diff"]) if m["diff"] else None)

# read every tab and return parsed questions plus a list of skipped rows with reasons
def load_for_preview(path):
    p=Path(path); ext=p.suffix.lower(); out=[]; skipped=[]; seen=set()
    if ext in (".xlsx",".xls"):
        for name,raw in pd.read_excel(p, sheet_name=None, header=None).items():
            _rows(_prep(raw), name, out, skipped, seen)
    elif ext==".tsv":
        _rows(_prep(pd.read_csv(p,sep="\t",header=None,dtype=object)), "tsv", out, skipped, seen)
    else:
        _rows(_prep(pd.read_csv(p,header=None,dtype=object)), "csv", out, skipped, seen)
    return out, skipped

# print every question with options labelled and the correct one or more starred
def print_questions(rows, skipped):
    print(f"parsed {len(rows)} questions; skipped {len(skipped)}\n")
    last=None
    for r in rows:
        if r["sheet"]!=last: print("\n#### sheet:", r["sheet"], "####"); last=r["sheet"]
        ans=set(r["answers"])
        print(f"\n[{r['qid']}]  {r['stem']}")
        for L in LETTERS:
            if L in r["options"]:
                print(f"   {'*' if L in ans else ' '} {L}. {r['options'][L]}")
        print(f"   correct: {', '.join(r['answers'])}")
    multi=sum(1 for r in rows if len(r["answers"])>1)
    five=sum(1 for r in rows if len(r["options"])>=5)
    print("\n"+"="*64)
    print(f"total parsed: {len(rows)}")
    print(f"  with multiple correct answers: {multi}")
    print(f"  with 5 options: {five}")
    if skipped:
        print(f"\nskipped {len(skipped)}:")
        for s in skipped[:40]: print(f"  (sheet {s['sheet']}) {s['reason']}: {s['preview']}")

# write the parsed questions to json so prototype_nav can read them directly instead of reparsing
def write_questions_json(rows, path):
    keys=("qid","stem","options","answer","answers","difficulty","sheet")
    payload=[{k:r[k] for k in keys if k in r} for r in rows]
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nwrote {len(payload)} questions to {path}")

In [4]:
rows, skipped = load_for_preview(QUESTIONS_FILE)
print_questions(rows, skipped)
write_questions_json(rows, QUESTIONS_JSON)

parsed 225 questions; skipped 98


#### sheet: General QMS ####

[QMS-1]  Where do you find SOPs?
     A. In a central lab binder
   * B. On the Quality SPN
     C. In a drawer at the workstation
     D. On the wiki
   correct: B

[QMS-2]  What version of an SOP should be used?
     A. Version 1.0 should always be used
     B. The version at the workstation
   * C. The version on the Quality SPN
     D. The version you were originally trained to use
   correct: C

[QMS-3]  How do you initiate the CAPA process?
     A. Complete the CAPA form on the Quality SPN and submit to QA
     B. Report a non-conformance to your supervisor
     C. Report a non-conformance to the QA Manager
   * D. Any of the above
   correct: D

[QMS-4]  What conditions count as a non-conformance?
   * A. Test results are at the high end of the expected range
     B. An SOP is not followed exactly
     C. Samples are swapped
     D. Equipment experiences an error while processing samples
   correct: A

[QMS-5]  Wha